In [0]:
%pip install kagglehub

In [0]:
dbutils.library.restartPython()

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nabihazahid/spotify-dataset-for-churn-analysis")

print("Path to dataset files:", path)

In [0]:
# Sprawdź jakie pliki są w katalogu
import os

files = os.listdir(path)
print(f"Files in dataset directory:")
for file in files:
    file_path = os.path.join(path, file)
    size = os.path.getsize(file_path)
    print(f"  - {file} ({size:,} bytes)")

In [0]:
# Wczytaj CSV do Spark DataFrame
# Kagglehub pobiera pliki lokalnie, więc musimy użyć pandas najpierw
import pandas as pd

csv_path = f"{path}/spotify_churn_dataset.csv"

# 1. Wczytaj lokalnie używając pandas
print("Loading CSV with pandas...")
df_pandas = pd.read_csv(csv_path)
print(f"Pandas DataFrame shape: {df_pandas.shape}")

# 2. Przekonwertuj pandas DataFrame na Spark DataFrame
print("\nConverting to Spark DataFrame...")
df_spotify = spark.createDataFrame(df_pandas)

print(f"\nNumber of rows: {df_spotify.count():,}")
print(f"Number of columns: {len(df_spotify.columns)}")
print(f"\nSchema:")
df_spotify.printSchema()
print(f"\nFirst 5 rows:")
display(df_spotify.limit(5))

## Analiza: Związek między typem subskrypcji a intensywnością korzystania ze Spotify

**Pytania badawcze:**
1. Jak typ subskrypcji wpływa na **częstotliwość przeskakiwania piosenek** (skip_rate)?
2. Jak typ subskrypcji wpływa na **liczbę przesłuchanych piosenek dziennie** (songs_played_per_day)?
3. Jak typ subskrypcji wpływa na **czas poświęcony na słuchanie** (listening_time)?

In [0]:
from pyspark.sql import functions as F

# Sprawdź rozkład typów subskrypcji
print("=" * 60)
print("ROZKŁAD TYPÓW SUBSKRYPCJI")
print("=" * 60)

subscription_dist = (df_spotify
    .groupBy("subscription_type")
    .agg(
        F.count("*").alias("liczba_uzytkownikow"),
        (F.count("*") / F.lit(df_spotify.count()) * 100).alias("procent")
    )
    .orderBy(F.desc("liczba_uzytkownikow")))

display(subscription_dist)

In [0]:
# Agregacje wg typu subskrypcji dla trzech kluczowych metryk
print("=" * 60)
print("STATYSTYKI WG TYPU SUBSKRYPCJI")
print("=" * 60)

subscription_stats = (df_spotify
    .groupBy("subscription_type")
    .agg(
        # Liczba użytkowników
        F.count("*").alias("liczba_uzytkownikow"),
        
        # Skip Rate (częstotliwość przeskakiwania)
        F.round(F.avg("skip_rate"), 3).alias("średni_skip_rate"),
        F.round(F.min("skip_rate"), 3).alias("min_skip_rate"),
        F.round(F.max("skip_rate"), 3).alias("max_skip_rate"),
        
        # Songs played per day (liczba piosenek dziennie)
        F.round(F.avg("songs_played_per_day"), 1).alias("średnia_liczba_piosenek_dziennie"),
        F.min("songs_played_per_day").alias("min_piosenek_dziennie"),
        F.max("songs_played_per_day").alias("max_piosenek_dziennie"),
        
        # Listening time (czas słuchania)
        F.round(F.avg("listening_time"), 1).alias("średni_czas_sluchania_min"),
        F.min("listening_time").alias("min_czas_sluchania"),
        F.max("listening_time").alias("max_czas_sluchania")
    )
    .orderBy("subscription_type"))

display(subscription_stats)

In [0]:
# Przygotuj uproszczoną tabelę do wizualizacji
visualization_data = (df_spotify
    .groupBy("subscription_type")
    .agg(
        F.round(F.avg("skip_rate"), 3).alias("avg_skip_rate"),
        F.round(F.avg("songs_played_per_day"), 1).alias("avg_songs_per_day"),
        F.round(F.avg("listening_time"), 1).alias("avg_listening_time")
    )
    .orderBy("subscription_type"))

print("Dane przygotowane do wizualizacji:")
display(visualization_data)

In [0]:
# Porównanie poszczególnych metryk
print("\n" + "=" * 60)
print("PORÓWNANIE METRYK WG SUBSKRYPCJI")
print("=" * 60)

# Ranking wg Skip Rate
print("\n1. SKIP RATE (częstotliwość przeskakiwania piosenek):")
print("   Niższy = lepiej (użytkownik bardziej zadowolony z muzyki)\n")
skip_ranking = (df_spotify
    .groupBy("subscription_type")
    .agg(F.round(F.avg("skip_rate"), 4).alias("avg_skip_rate"))
    .orderBy("avg_skip_rate"))
for row in skip_ranking.collect():
    print(f"   {row.subscription_type:12} -> {row.avg_skip_rate:.4f} (30.{int(row.avg_skip_rate*10000-3000)}%)")

# Ranking wg Songs per day
print("\n2. LICZBA PIOSENEK DZIENNIE:")
print("   Więcej = bardziej aktywny użytkownik\n")
songs_ranking = (df_spotify
    .groupBy("subscription_type")
    .agg(F.round(F.avg("songs_played_per_day"), 2).alias("avg_songs"))
    .orderBy(F.desc("avg_songs")))
for row in songs_ranking.collect():
    print(f"   {row.subscription_type:12} -> {row.avg_songs:.1f} piosenek/dzień")

# Ranking wg Listening time
print("\n3. CZAS SŁUCHANIA:")
print("   Więcej = większe zaangażowanie\n")
time_ranking = (df_spotify
    .groupBy("subscription_type")
    .agg(F.round(F.avg("listening_time"), 1).alias("avg_time"))
    .orderBy(F.desc("avg_time")))
for row in time_ranking.collect():
    print(f"   {row.subscription_type:12} -> {row.avg_time:.1f} minut ({row.avg_time/60:.1f}h)")

In [0]:
# Oblicz odchylenie standardowe i wariancję dla metryk
print("\n" + "=" * 60)
print("WARIANCJA I ODCHYLENIE STANDARDOWE")
print("=" * 60)

std_stats = (df_spotify
    .groupBy("subscription_type")
    .agg(
        F.round(F.stddev("skip_rate"), 4).alias("stddev_skip_rate"),
        F.round(F.stddev("songs_played_per_day"), 2).alias("stddev_songs"),
        F.round(F.stddev("listening_time"), 2).alias("stddev_listening_time")
    )
    .orderBy("subscription_type"))

print("\nOdchylenie standardowe (większe = większa różnorodność zachowań):")
display(std_stats)

# Współczynnik zmienności (CV) - pokazuje względną zmienność
print("\nWspółczynnik zmienności (CV = stddev/mean):")
cv_stats = (df_spotify
    .groupBy("subscription_type")
    .agg(
        F.round((F.stddev("skip_rate") / F.avg("skip_rate")) * 100, 2).alias("cv_skip_rate_%"),
        F.round((F.stddev("songs_played_per_day") / F.avg("songs_played_per_day")) * 100, 2).alias("cv_songs_%"),
        F.round((F.stddev("listening_time") / F.avg("listening_time")) * 100, 2).alias("cv_listening_time_%")
    )
    .orderBy("subscription_type"))

display(cv_stats)

## 📊 Kluczowe wnioski z analizy

### 1️⃣ **Brak istotnych różnic między typami subskrypcji**

Analiza wykazała **zaskakująco małe różnice** w intensywności użytkowania między różnymi typami subskrypcji:

#### ⚠️ Skip Rate (częstotliwość przeskakiwania):
* **Premium**: 0.297 (29.7%)
* **Family**: 0.300 (30.0%)
* **Free**: 0.301 (30.1%)
* **Student**: 0.303 (30.3%)
* **Różnica**: zaledwie **0.6 punktu procentowego** między najniższym a najwyższym!

#### 🎵 Liczba piosenek dziennie:
* **Student**: 51.2 piosenek
* **Family**: 50.4 piosenek
* **Premium**: 49.7 piosenek
* **Free**: 49.2 piosenek
* **Różnica**: tylko **2 piosenki** (4%) między ekstremami

#### ⏱️ Czas słuchania:
* **Premium**: 155.5 minut (2.59h)
* **Free**: 155.0 minut (2.58h)
* **Student**: 154.5 minut (2.58h)
* **Family**: 151.0 minut (2.52h)
* **Różnica**: tylko **4.5 minuty** (3%) między ekstremami

---

### 2️⃣ **Interpretacja wyników**

✅ **Pozytywne obserwacje:**
* Wszystkie grupy użytkowników są **jednakowo zaangażowane**
* Typ subskrypcji **nie determinuje** zachowania użytkownika
* Użytkownicy darmowi są tak samo aktywni jak płatni

⚠️ **Potencjalne problemy:**
* **Strategia monetyzacji** - użytkownicy Free nie są mniej aktywni, więc co motywuje do upgrade'u?
* **Value proposition** - czy subskrypcje premium oferują wystarczającą wartość?
* Skip rate ~30% **we wszystkich grupach** to wysoki wskaźnik - problem z rekomendacjami?

---

### 3️⃣ **Rekomendacje biznesowe**

1. **Zbadaj inne czynniki** wpływające na churn - skoro intensywność jest podobna, problem leży gdzie indziej
2. **Ulepsz algorytmy rekomendacji** - 30% skip rate sugeruje, że użytkownicy często nie znajdują odpowiedniej muzyki
3. **Zróżnicuj value proposition** - subskrypcje płatne muszą oferować wyraźne korzyści poza brakiem reklam
4. **Przeanalizuj jakość słuchania** - może płatni użytkownicy słuchają wyższej jakości audio, ale to nie widoczne w tych metrykach

---

### 4️⃣ **Ograniczenia analizy**

* Analiza oparta **tylko na agregowanych średnich** - nie uwzględnia rozkładów wewnętrznych
* Brak analizy **trendów czasowych** - czy zachowania zmieniają się w czasie?
* Brak **analizy jakościowej** - jakich gatunków słuchają różne grupy?
* Brak **segmentacji geograficznej** i demograficznej

In [0]:
# Analiza korelacji między zmiennymi
print("\n" + "=" * 60)
print("ANALIZA KORELACJI")
print("=" * 60)

from pyspark.sql.functions import corr

# Korelacje ogólne (dla całego datasetu)
print("\nKorelacje między kluczowymi metrykami (cały dataset):")
print("\n1. Skip Rate vs Songs per Day:")
corr_skip_songs = df_spotify.select(corr("skip_rate", "songs_played_per_day")).collect()[0][0]
print(f"   Korelacja: {corr_skip_songs:.4f}")
if abs(corr_skip_songs) < 0.1:
    print("   → Brak związku (korelacja bliska 0)")

print("\n2. Skip Rate vs Listening Time:")
corr_skip_time = df_spotify.select(corr("skip_rate", "listening_time")).collect()[0][0]
print(f"   Korelacja: {corr_skip_time:.4f}")
if abs(corr_skip_time) < 0.1:
    print("   → Brak związku (korelacja bliska 0)")

print("\n3. Songs per Day vs Listening Time:")
corr_songs_time = df_spotify.select(corr("songs_played_per_day", "listening_time")).collect()[0][0]
print(f"   Korelacja: {corr_songs_time:.4f}")
if corr_songs_time > 0.5:
    print("   → Umiarkowany pozytywny związek")
elif corr_songs_time > 0.3:
    print("   → Słaby pozytywny związek")
else:
    print("   → Bardzo słaby związek")

# Porównanie zachowań użytkowników offline vs online
print("\n" + "=" * 60)
print("PORÓWNANIE: OFFLINE vs ONLINE LISTENING")
print("=" * 60)

offline_comparison = (df_spotify
    .groupBy("offline_listening")
    .agg(
        F.count("*").alias("liczba_uzytkownikow"),
        F.round(F.avg("skip_rate"), 3).alias("avg_skip_rate"),
        F.round(F.avg("songs_played_per_day"), 1).alias("avg_songs_per_day"),
        F.round(F.avg("listening_time"), 1).alias("avg_listening_time")
    )
    .orderBy("offline_listening"))

print("\n0 = tylko online, 1 = używa offline")
display(offline_comparison)

---

## 🎯 OSTATECZNE PODSUMOWANIE

### Główne odkrycie: **TYP SUBSKRYPCJI NIE WPŁYWA NA INTENSYWNOŚĆ UŻYTKOWANIA**

#### 📊 Zestawienie liczbowe:

| Typ Subskrypcji | Użytkownicy | Skip Rate | Piosenki/dzień | Czas słuchania (min) |
|-----------------|-------------|-----------|------------------|------------------------|
| **Premium**     | 2,115 (26%) | 0.297     | 49.7             | 155.5                  |
| **Free**        | 2,018 (25%) | 0.301     | 49.2             | 155.0                  |
| **Student**     | 1,959 (24%) | 0.303     | 51.2             | 154.5                  |
| **Family**      | 1,908 (24%) | 0.300     | 50.4             | 151.0                  |
| **RÓŻNICA MAX** | -           | **0.006** | **2.0**          | **4.5**                |

---

### 🔍 Co oznaczają te wyniki?

#### ✅ **Pozytywy:**
1. **Wysoka aktywność użytkowników darmowych** - nie są "gorszymi" użytkownikami
2. **Stabilna baza użytkowników** - wszystkie grupy jednakowo zaangażowane
3. **Demokratyzacja platformy** - jakość usługi niezależna od płatności

#### ⚠️ **Pytania strategiczne:**
1. **Dlaczego użytkownicy upgrade'ują do premium?** - jeśli zachowania są identyczne
2. **Czy premium ma wystarczającą wartość?** - co poza brakiem reklam?
3. **30% skip rate** - czy algorytmy rekomendacji są wystarczająco dobre?

---

### 🚀 Następne kroki analizy:

1. **Segmentacja demograficzna** (wiek, płeć, kraj)
2. **Analiza churn** - co faktycznie powoduje rezygnację?
3. **Analiza jakościowa** - gatunki muzyczne, playlisty, artyści
4. **Trendy czasowe** - jak zachowania zmieniają się w czasie?
5. **Analiza konwersji** - co skłania Free → Premium?

---

### 📊 Dodatkowe obserwacje:

* **Brak korelacji** między skip_rate, songs_per_day i listening_time (r ≈ 0)
* **75% użytkowników** korzysta z funkcji offline (5,982 vs 2,018)
* **Wysoka wariancja** wewnątrz grup (CV ~55-58%) - użytkownicy bardzo różni pomimo tej samej subskrypcji

In [0]:
# Stwórz temp view dla wizualizacji
visualization_data.createOrReplaceTempView("subscription_metrics")

# Dodaj także szczegółowe statystyki
subscription_stats.createOrReplaceTempView("subscription_stats_detailed")

print("Temp views utworzone:")
print("  - subscription_metrics (dla wykresów)")
print("  - subscription_stats_detailed (szczegółowe statystyki)")

In [0]:
%sql
SELECT 
  subscription_type,
  avg_skip_rate,
  avg_songs_per_day,
  avg_listening_time
FROM subscription_metrics
ORDER BY subscription_type

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

---

## 🛠️ Metodologia analizy

### Dataset:
* **Źródło:** Kaggle - Spotify Dataset for Churn Analysis
* **Rozmiar:** 8,000 użytkowników
* **Format:** CSV (391 KB)
* **Kolumny:** 12 (user_id, gender, age, country, subscription_type, listening_time, songs_played_per_day, skip_rate, device_type, ads_listened_per_week, offline_listening, is_churned)

### Narzędzia:
* **Apache Spark (PySpark)** - przetwarzanie danych
* **Pandas** - import lokalnego pliku CSV
* **Databricks Notebooks** - środowisko analityczne
* **SQL** - zapytania agregujące

### Metody statystyczne:
* **Statystyki opisowe:** średnia, min, max
* **Miary zmienności:** odchylenie standardowe, współczynnik zmienności (CV)
* **Korelacja Pearsona:** analiza związków między zmiennymi
* **Agregacje grupowe:** GROUP BY wg subscription_type

### Kroki analizy:
1. Załadowanie danych z Kaggle (kagglehub)
2. Konwersja pandas → Spark DataFrame
3. Eksploracja rozkładu typów subskrypcji
4. Agregacje wg subscription_type dla 3 kluczowych metryk
5. Analiza wariancji i odchyleń standardowych
6. Analiza korelacji między zmiennymi
7. Porównanie offline vs online listening
8. Wizualizacje wyników

### Kluczowe metryki:
* **Skip Rate** - % piosenek pominiętych (0-1)
* **Songs Played Per Day** - liczba piosenek dziennie (1-99)
* **Listening Time** - czas słuchania w minutach (10-299)

---

**Data analizy:** 2026-06-23  
**Analityk:** Databricks Assistant  
**Notebook:** [Spotify Subscription Analysis](#notebook/2566943964963674)